In [21]:
import pandas as pd
df = pd.read_csv('for_export_philippine_elections.csv')

authors = pd.read_csv('well_known_authors_philippine_elections.csv')

df.head()

,pseudo_id,text,retweetCount,replyCount,likeCount,quoteCount,viewCount,bookmarkCount,createdAt,lang,isReply,pseudo_conversationId,pseudo_inReplyToUsername,pseudo_author_userName,author_isBlueVerified
0,224179509746025,not a fan of Brent pero ano ba 'yang Big Winne...,0,0,0,0,5,0,2025-05-31 16:22:41+00:00,tl,False,224179509746025,NaN,733727272962306,False
1,998635197269390,wala talagang aasahan sa mga botante JUSKOPO,0,0,0,0,7,0,2025-05-31 16:21:56+00:00,tl,False,998635197269390,NaN,924044017517801,False
2,954085072967672,@975452594351818 Magiging presidente para guma...,0,0,0,0,0,0,2025-05-31 16:21:03+00:00,tl,True,725047262887831,9.754526e+14,685540093851332,False
3,90429221872722,Bumoto kahit walang idea kung ano to. Promise ...,0,0,0,0,2,0,2025-05-31 16:20:20+00:00,tl,False,90429221872722,NaN,897590766311451,False
4,503936424634617,alam mo talagang wikla duo ang bet ng CVs kase...,0,0,0,0,1,0,2025-05-31 16:19:10+00:00,tl,False,503936424634617,NaN,887555520780376,False


In [23]:
edges = df.groupby(['pseudo_author_userName','pseudo_inReplyToUsername']).size().reset_index(name='weight')
edges.sort_values('weight',ascending=False).head()

,pseudo_author_userName,pseudo_inReplyToUsername,weight
58450,982646580214281,9.826466e+14,418
25696,430526661831933,4.305267e+14,379
58148,977770615637911,9.777706e+14,322
35433,589029215591253,5.890292e+14,300
24268,403780272519270,4.037803e+14,269


In [ ]:
%matplotlib inline
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import warnings
from collections import Counter
import sys
import os
from __future__ import division
plt.rcParams["figure.figsize"] = (20,10)
import tqdm as tqdm

warnings.filterwarnings('ignore')
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:80% !important; }</style>"))

G = nx.from_pandas_edgelist(edges, 
                            source='pseudo_author_userName',
                            target='pseudo_inReplyToUsername',
                            edge_attr='weight',
                            create_using=nx.DiGraph())
print(G)
in_degrees = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)
out_degrees = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)

# top 10 users
print("All in-degrees:", in_degrees[:10])
print("All out-degrees:", out_degrees[:10])   

DiGraph with 36609 nodes and 59519 edges
All in-degrees: [(403780272519270.0, 1816), (460436796410133.0, 834), (589029215591253.0, 735), (623361697158128.0, 565), (901946442613399.0, 557), (776109641341820.0, 546), (36502604678147.0, 516), (325680252986141.0, 501), (325002683734926.0, 481), (556776310795431.0, 478)]
All out-degrees: [(439483351890134.0, 117), (500445945093556.0, 97), (602057431449805.0, 88), (810704260922273.0, 85), (245705656757626.0, 84), (554423716436549.0, 82), (925279596835042.0, 81), (88987228372981.0, 81), (92125075573352, 76), (453836684166219.0, 76)]


In [56]:
author_usernames = dict(zip(
    authors['obfuscated_userName'].astype(str).str[1:], 
    authors['author_userName']
))
print(author_usernames)

for i in range(len(in_degrees)):
    id = str(in_degrees[i][0])
    if id in author_usernames:
        in_degrees[i] = (author_usernames[id], in_degrees[i][1])
for i in range(len(out_degrees)):
    id = str(out_degrees[i][0])
    if id in author_usernames:
        out_degrees[i] = (author_usernames[id], out_degrees[i][1])

# top 10 users
print("All in-degrees:", in_degrees[:10])
print("All out-degrees:", out_degrees[:10])

{'447129996589185': '__nisanjo', '502848624539866': '_djchacha', '894831624795349': '_GaiaPoly', '344694405381762': 'ABSCBN', '403780272519270': 'ABSCBNNews', '34859533045326': 'ABSCBNpr', '456793039343990': 'AFP', '876575320724228': 'AHOF__GRANDMA', '347280075511207': 'akazukinchan2x', '11503056031366': 'akoposimarcelo', '526730790366201': 'AltStarMagic', '467161425997003': 'alvinelchico', '802585864361733': 'amaxtriton', '208287835608527': 'ANCALERTS', '936960220706872': 'apangilinan', '70047054001669': 'Arshiethromycin', '712192195397021': 'AskPerplexity', '575306104619656': 'ateneodemanilau', '328126839578840': 'attyharryroque', '894416324110752': 'AttyLeiladeLima', '730798648352396': 'AttyMatula', '951392445303700': 'bamaquino', '768795647400867': 'BangkoSentral', '33643637499439': 'barrygutierrez3', '326683475544033': 'bayanmo', '942501146785377': 'BbMaharlika', '979339868045075': 'beabinene', '69469140131426': 'bgyo_ph', '607504780903547': 'BiniJhoannaOFC', '722855629086973': 'b

In [57]:
centrality = nx.betweenness_centrality(G, k=None, normalized=True, weight=None, endpoints=False, seed=None)

for i in range(len(centrality)):
    id = str(centrality[i][0])
    if id in author_usernames:
        centrality[i] = (author_usernames[id], centrality[i][1])

print("Betweeness centrality:", centrality[:10])

KeyboardInterrupt: 